In [20]:
from google import genai
from dotenv import load_dotenv
load_dotenv()
import os

gemini_client=genai.Client(api_key=os.getenv("GEMINI_KEY"))

In [21]:
import numpy as np

def cosine(a,b):
    a=np.array(a)
    b=np.array(b)
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

In [22]:
ananya_txt = ""
vikram_txt = ""
meera_txt = ""
arjun_txt = ""
priya_txt = ""
karthik_txt = ""

with open("professor1.txt", "r") as f:
    professor1_txt = f.read()

with open("professor2.txt", "r") as f:
    professor2_txt = f.read()

with open("professor3.txt", "r") as f:
    professor3_txt = f.read()

with open("professor4.txt", "r") as f:
    professor4_txt = f.read()

with open("professor5.txt", "r") as f:
    professor5_txt = f.read()

with open("professor6.txt", "r") as f:
    professor6_txt = f.read()


print(professor1_txt)
print(professor2_txt)
print(professor3_txt)
print(professor4_txt)
print(professor5_txt)
print(professor6_txt)

Name: Dr. Ananya Rao

Department: Computer Science and Engineering

Designation: Associate Professor

Experience: 14 Years

Highest Qualification: Ph.D. in Computer Science, Indian Institute of Technology Madras

Research Interests:
Natural Language Processing (NLP), Large Language Models (LLMs), Information Retrieval, Question Answering Systems, Machine Translation, Conversational AI, Text Summarization, Multilingual Artificial Intelligence

Research Keywords:
NLP, LLM, Transformer Models, Retrieval-Augmented Generation (RAG), Information Retrieval, BERT, GPT, Machine Translation, Semantic Search, Embeddings

Current Research Projects:
1. AI-powered multilingual educational assistant for Indian languages.
2. Retrieval-Augmented Question Answering System for university academic resources.
3. Low-resource machine translation between Telugu and English.
4. Conversational AI assistant for higher education.

Selected Publications:
1. Dense Retrieval Techniques for Educational Question Answ

In [23]:
professor1_embed = gemini_client.models.embed_content(
    model="models/gemini-embedding-001",
    contents=[professor1_txt]
)

professor2_embed = gemini_client.models.embed_content(
    model="models/gemini-embedding-001",
    contents=[professor2_txt]
)

professor3_embed = gemini_client.models.embed_content(
    model="models/gemini-embedding-001",
    contents=[professor3_txt]
)

professor4_embed = gemini_client.models.embed_content(
    model="models/gemini-embedding-001",
    contents=[professor4_txt]
)

professor5_embed = gemini_client.models.embed_content(
    model="models/gemini-embedding-001",
    contents=[professor5_txt]
)

professor6_embed = gemini_client.models.embed_content(
    model="models/gemini-embedding-001",
    contents=[professor6_txt]
)

In [24]:
professor1_embed = professor1_embed.embeddings[0].values
professor2_embed = professor2_embed.embeddings[0].values
professor3_embed = professor3_embed.embeddings[0].values
professor4_embed = professor4_embed.embeddings[0].values
professor5_embed = professor5_embed.embeddings[0].values
professor6_embed = professor6_embed.embeddings[0].values

In [25]:
import chromadb

chromadb_client = chromadb.PersistentClient(path="./chroma_db")

collection = chromadb_client.get_or_create_collection(
    name="faculty_profiles"
)

print("Collection Created:", collection.name)

Collection Created: faculty_profiles


In [26]:
all_professors = [
    professor1_txt,
    professor2_txt,
    professor3_txt,
    professor4_txt,
    professor5_txt,
    professor6_txt
]

In [27]:
ids = []

for i in range(len(all_professors)):
    ids.append(str(i))

In [28]:
print(ids)

['0', '1', '2', '3', '4', '5']


In [29]:
all_embeddings = [
    professor1_embed,
    professor2_embed,
    professor3_embed,
    professor4_embed,
    professor5_embed,
    professor6_embed
]

In [30]:
collection.add(
    ids=ids,
    documents=all_professors,
    embeddings=all_embeddings
)

In [31]:
user_query = "I am interested in Artificial Intelligence and Machine Learning. Which professor should I contact?"

In [32]:
query_embed = gemini_client.models.embed_content(
    model="models/gemini-embedding-001",
    contents=[user_query]
)

query_embed = query_embed.embeddings[0].values

In [33]:
print(len(query_embed))

3072


In [48]:
results = collection.query(
    query_embeddings=[query_embed],
    n_results=3
)

In [49]:
print("Retrieved IDs:")
print(results["ids"])

print("\nDistances:")
print(results["distances"])

Retrieved IDs:
[['5', '1', '0']]

Distances:
[[0.6773979067802429, 0.7230738401412964, 0.7448145747184753]]


In [50]:
print(results)

{'ids': [['5', '1', '0']], 'embeddings': None, 'documents': [['Name: Dr. Karthik Menon\n\nDepartment: Computer Science and Engineering\n\nDesignation: Professor\n\nExperience: 17 Years\n\nHighest Qualification: Ph.D. in Machine Learning, IIT Kanpur\n\nResearch Interests:\nMachine Learning, Explainable AI, Reinforcement Learning, Predictive Modeling, Artificial Intelligence, Decision Support Systems\n\nResearch Keywords:\nMachine Learning, Explainable AI, Reinforcement Learning, Neural Networks, Classification, Regression, Model Interpretability, AI Ethics\n\nCurrent Research Projects:\n1. Explainable AI for Financial Risk Prediction.\n2. Reinforcement Learning for Autonomous Decision Making.\n3. Responsible AI Frameworks.\n4. AI-assisted Decision Support Systems.\n\nSelected Publications:\n1. Explainable Machine Learning for Healthcare (2025)\n2. Reinforcement Learning in Autonomous Systems (2024)\n3. Interpretable AI Models for Financial Analytics (2023)\n\nCourses Taught:\nMachine Le

In [51]:
print(results["ids"])

[['5', '1', '0']]


In [52]:
print(results["documents"][0][0])

Name: Dr. Karthik Menon

Department: Computer Science and Engineering

Designation: Professor

Experience: 17 Years

Highest Qualification: Ph.D. in Machine Learning, IIT Kanpur

Research Interests:
Machine Learning, Explainable AI, Reinforcement Learning, Predictive Modeling, Artificial Intelligence, Decision Support Systems

Research Keywords:
Machine Learning, Explainable AI, Reinforcement Learning, Neural Networks, Classification, Regression, Model Interpretability, AI Ethics

Current Research Projects:
1. Explainable AI for Financial Risk Prediction.
2. Reinforcement Learning for Autonomous Decision Making.
3. Responsible AI Frameworks.
4. AI-assisted Decision Support Systems.

Selected Publications:
1. Explainable Machine Learning for Healthcare (2025)
2. Reinforcement Learning in Autonomous Systems (2024)
3. Interpretable AI Models for Financial Analytics (2023)

Courses Taught:
Machine Learning
Artificial Intelligence
Deep Learning
Reinforcement Learning
Data Science

Research 

In [53]:
retrieved_docs = results["documents"][0]

In [54]:
context = "\n\n".join(retrieved_docs)

In [55]:
print(context)

Name: Dr. Karthik Menon

Department: Computer Science and Engineering

Designation: Professor

Experience: 17 Years

Highest Qualification: Ph.D. in Machine Learning, IIT Kanpur

Research Interests:
Machine Learning, Explainable AI, Reinforcement Learning, Predictive Modeling, Artificial Intelligence, Decision Support Systems

Research Keywords:
Machine Learning, Explainable AI, Reinforcement Learning, Neural Networks, Classification, Regression, Model Interpretability, AI Ethics

Current Research Projects:
1. Explainable AI for Financial Risk Prediction.
2. Reinforcement Learning for Autonomous Decision Making.
3. Responsible AI Frameworks.
4. AI-assisted Decision Support Systems.

Selected Publications:
1. Explainable Machine Learning for Healthcare (2025)
2. Reinforcement Learning in Autonomous Systems (2024)
3. Interpretable AI Models for Financial Analytics (2023)

Courses Taught:
Machine Learning
Artificial Intelligence
Deep Learning
Reinforcement Learning
Data Science

Research 

In [56]:
prompt = f"""
You are an AI Faculty Assistant for a university.

Use ONLY the information provided in the context below.

Your responsibilities are:
1. Recommend professors based on student interests.
2. Suggest relevant courses taught by the professor.
3. Mention research interests when relevant.
4. Mention projects or publications if they help answer the question.
5. If the answer is not available in the context, clearly say you don't have enough information.

Context:
{context}

Student Question:
{user_query}

Provide a clear, friendly, and well-structured answer.
"""

In [57]:
response = gemini_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

In [58]:
print(response.text)

Based on your interest in Artificial Intelligence and Machine Learning, I recommend contacting **Dr. Karthik Menon**.

Here's why he would be an excellent fit:

*   **Research Interests:** Dr. Menon's primary research interests directly align with yours, including **Machine Learning**, **Artificial Intelligence**, Explainable AI, Reinforcement Learning, and Predictive Modeling.
*   **Courses Taught:** He teaches core courses such as **Machine Learning** and **Artificial Intelligence**, along with Deep Learning and Reinforcement Learning.
*   **Student Guidance:** He is available for student guidance in **Machine Learning** and **Artificial Intelligence**, among other related areas. His biography states that "Students interested in core machine learning algorithms, model interpretability, and intelligent decision-making systems frequently work under his guidance."
*   **Relevant Projects & Publications:** His current research projects include "Explainable AI for Financial Risk Predictio